In [ ]:
from pathlib import Path
import base64,hashlib,io,zipfile,os,subprocess,shutil
from IPython.display import display,FileLink
work=Path('/kaggle/working')
project=work/'ns64_ets_project'
if project.exists():raise FileExistsError('Preserve this attempt. Use a new session or reviewed new workspace for retry.')
project.mkdir()
raw=base64.b64decode('...')
assert hashlib.sha256(raw).hexdigest()=='f26cf4bcd6f141f015156ed25d38f4458e3459001abd82403a8b04550d6861ca'
with zipfile.ZipFile(io.BytesIO(raw)) as z:z.extractall(project)
source_matches=[]
for directory,folders,files in os.walk('/kaggle/input'):
    folders[:]=[n for n in folders if n not in ('competitions','train','test') and not n.startswith('image-matching-challenge-')]
    if 'ns64_pilot_source.zip.bin' in files:source_matches.append(Path(directory)/'ns64_pilot_source.zip.bin')
assert len(source_matches)==1,source_matches
target=project/'configs/baselines/top10_reproduction_v1/ns64_pilot_source.zip.bin'
assert hashlib.sha256(source_matches[0].read_bytes()).hexdigest()=='c6035c596a484331ebfaf8d3b6368b4a32acb00b3bebe670a8215c78b6c12549'
shutil.copyfile(source_matches[0],target)
collections=[p for p in [
    Path('/kaggle/input/competitions/image-matching-challenge-2025-ongoing/train/ETs'),
    Path('/kaggle/input/image-matching-challenge-2025-ongoing/train/ETs'),
] if p.is_dir()]
assert len(collections)==1,collections
python=work/'ns64_env/bin/python'
receipt=work/'imc_ns64_dependency_receipt.zip'
assert python.is_file() and receipt.is_file(),'Retain the validated session and receipt'
assert hashlib.sha256(receipt.read_bytes()).hexdigest()=='3225dda6d7a02d1d6008a672c13695e8c45941b9f78087d9644b8c04a262bf6b'
env=os.environ.copy()
env.update(MPLBACKEND='Agg',HF_HUB_OFFLINE='1',TRANSFORMERS_OFFLINE='1',SCENE_SPACE_DIR_PERSISTENT='yes')
attempt=work/'ns64_real_ets_pilot'
print('Starting published ns64 ETs pipeline. Budget 1800 seconds.',flush=True)
result=subprocess.run([str(python),str(project/'src/baselines/top10_reproduction/ets_worker.py'),
    '--work',str(attempt),'--images',str(collections[0]),'--receipt',str(receipt)],env=env)
print('Launcher exit',result.returncode)
archive=attempt/'imc_ns64_ets_pilot.zip'
if archive.is_file():
    with zipfile.ZipFile(archive) as z:
        import json
        print(json.loads(z.read('result.json'))['status'])
    display(FileLink(str(archive)))
else:print('Inspect the traceback and preserve this working session.')
